<a href="https://colab.research.google.com/github/jhhlim/LLMFundamentals/blob/main/Jason_Lim_hw_6a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 6a — LLM Fundamentals (UCSC Extension)
# Homework 6a: Practice LLM Fine-Tuning (Sentiment Analysis)

**Jason Lim**

## Assignment
**6a)** In your Sentiment Analysis homework example, implement fine-tuning based on class lab 6a. Do you see accuracy improvements?

This notebook applies the **Hugging Face `Trainer` fine-tuning pipeline from class lab 6a** (Hands-On LLM Chapter 11: *Fine-tuning Representation Models for Classification*) to the **same Amazon product-review dataset used in Homework 3a**.

## Dataset (from Homework 3a)
**UCI Sentiment Labelled Sentences — Amazon cell phone / product reviews**  
https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences

- 1,000 labeled review sentences
- `1` = positive, `0` = negative
- Homework 3a VADER baseline accuracy on the full 1,000 sentences: **0.768**

## What class lab 6a does (and what we reuse)
| Lab 6a piece | This homework |
|---|---|
| `bert-base-cased` + classification head | same |
| `AutoTokenizer` + `DataCollatorWithPadding` | same |
| Hugging Face `Trainer` / `TrainingArguments` | same |
| F1 via `evaluate` | F1 **and** accuracy (6a asks about accuracy) |
| `rotten_tomatoes` movie reviews | **Amazon product reviews** from HW 3a |
| Lab defaults: `lr=2e-5`, `batch=16`, `epochs=1`, `weight_decay=0.01` | same starting point |

## Runtime
Use a GPU in Colab: **Runtime → Change runtime type → T4 GPU**. CPU works on this small dataset, but is slower.


## Step 0 — Install + imports

In [ ]:
%%capture
!pip install -q "transformers>=4.38.2" "datasets>=2.18.0,<3" evaluate accelerate vaderSentiment pandas matplotlib seaborn scikit-learn


In [ ]:
import inspect
import os
import random
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
sns.set_theme(style="whitegrid")
%matplotlib inline

SEED = 42
MODEL_ID = "bert-base-cased"
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Transformers Trainer uses processing_class:", "processing_class" in inspect.signature(Trainer.__init__).parameters)


In [ ]:
def make_training_args(output_dir, **kwargs):
    '''Build TrainingArguments compatible with both older and newer transformers.'''
    sig = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"

    defaults = dict(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=1,
        weight_decay=0.01,
        save_strategy="epoch",
        logging_strategy="epoch",
        report_to="none",
        seed=SEED,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
        overwrite_output_dir=True,
    )
    defaults[eval_key] = "epoch"
    defaults.update(kwargs)

    # Drop kwargs the installed transformers version does not accept.
    filtered = {k: v for k, v in defaults.items() if k == "output_dir" or k in sig}
    return TrainingArguments(**filtered)


def make_trainer(model, args, train_dataset, eval_dataset, tokenizer, data_collator, compute_metrics):
    '''Trainer() compatibility wrapper: tokenizer vs processing_class.'''
    kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    sig = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in sig:
        kwargs["processing_class"] = tokenizer
    else:
        kwargs["tokenizer"] = tokenizer
    return Trainer(**kwargs)


accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    '''Accuracy + F1, following class lab 6a (Hands-On LLM Ch. 11 uses F1).'''
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=predictions, references=labels)["f1"],
    }


def plot_confusion(y_true, y_pred, title, labels=("negative", "positive")):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return cm


## Step 1 — Load the Homework 3a Amazon reviews and split

VADER does not train, so Homework 3a scored all 1,000 sentences. Fine-tuning needs a held-out test set, so we use a **stratified 80/20 split** (`seed=42`). VADER is re-scored on the **same test split** for a fair accuracy comparison.


In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"

with urlopen(DATA_URL) as resp:
    zf = ZipFile(BytesIO(resp.read()))

with zf.open("sentiment labelled sentences/amazon_cells_labelled.txt") as f:
    rows = []
    for line in f.read().decode("utf-8").splitlines():
        text, label = line.rsplit("\t", 1)
        rows.append({"text": text.strip(), "label": int(label)})

df = pd.DataFrame(rows)
print("Full dataset shape:", df.shape)
print(df["label"].value_counts().sort_index().rename({0: "negative (0)", 1: "positive (1)"}))
display(df.head(8))

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print("\nTrain:", train_df.shape, "Test:", test_df.shape)
print("Train labels:\n", train_df["label"].value_counts().sort_index().to_string())
print("Test labels:\n", test_df["label"].value_counts().sort_index().to_string())

raw_datasets = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    }
)


## Step 2 — VADER baseline on the same test split

Homework 3a used the class convention `compound >= 0 → positive`. We keep that rule so the baseline matches the earlier notebook.


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_predict(text: str) -> int:
    return int(analyzer.polarity_scores(text)["compound"] >= 0)

vader_test_pred = test_df["text"].map(vader_predict).to_numpy()
vader_test_true = test_df["label"].to_numpy()

vader_acc = accuracy_score(vader_test_true, vader_test_pred)
vader_f1 = f1_score(vader_test_true, vader_test_pred)

print("VADER on held-out test split")
print("Accuracy:", round(vader_acc, 4))
print("F1:", round(vader_f1, 4))
print()
print(classification_report(vader_test_true, vader_test_pred, target_names=["negative", "positive"]))
plot_confusion(vader_test_true, vader_test_pred, "VADER — test split")


## Step 3 — Tokenize like class lab 6a

Lab pattern: `bert-base-cased` tokenizer, truncate, and pad dynamically with `DataCollatorWithPadding`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
    '''Tokenize input data (class lab 6a).'''
    return tokenizer(examples["text"], truncation=True)

tokenized_train = raw_datasets["train"].map(preprocess_function, batched=True)
tokenized_test = raw_datasets["test"].map(preprocess_function, batched=True)

print(tokenizer(train_df.loc[0, "text"]))
print("Example tokens:", tokenizer.convert_ids_to_tokens(tokenizer(train_df.loc[0, "text"])["input_ids"][:24]))


## Step 4 — Fine-tune BERT with the class-lab Trainer settings

Class lab 6a `TrainingArguments`:

- `learning_rate=2e-5`
- `per_device_train_batch_size=16`
- `per_device_eval_batch_size=16`
- `num_train_epochs=1`
- `weight_decay=0.01`
- `save_strategy="epoch"`
- `report_to="none"`

We add epoch-level evaluation so we can report **accuracy and F1** on the test split after training.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

lab_args = make_training_args(
    "model/hw6a_lab_default",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
)

trainer = make_trainer(
    model=model,
    args=lab_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_output = trainer.train()
print(train_output)

lab_eval = trainer.evaluate()
print("\nLab-default fine-tune — test metrics:")
for k, v in lab_eval.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")


## Step 5 — Classification report for the fine-tuned model

In [ ]:
pred_output = trainer.predict(tokenized_test)
ft_pred = np.argmax(pred_output.predictions, axis=-1)
ft_true = np.array(tokenized_test["label"])

print("Fine-tuned BERT (lab defaults) on test split")
print("Accuracy:", round(accuracy_score(ft_true, ft_pred), 4))
print("F1:", round(f1_score(ft_true, ft_pred), 4))
print()
print(classification_report(ft_true, ft_pred, target_names=["negative", "positive"]))
plot_confusion(ft_true, ft_pred, "Fine-tuned BERT (lab defaults) — test split")


## Step 6 — Freeze layers (class lab 6a comparison)

The lab also trains a version where **only the classification head is trainable**. That is cheaper and tests how much of the gain comes from adapting BERT vs. just fitting a linear head on frozen features.


In [ ]:
frozen_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

for name, param in frozen_model.named_parameters():
    if name.startswith("classifier"):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in frozen_model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

frozen_args = make_training_args(
    "model/hw6a_frozen_head",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
)

frozen_trainer = make_trainer(
    model=frozen_model,
    args=frozen_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
frozen_trainer.train()
frozen_eval = frozen_trainer.evaluate()
print("\nFrozen-encoder fine-tune — test metrics:")
for k, v in frozen_eval.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")


## Step 7 — Do we see accuracy improvements?

In [ ]:
compare = pd.DataFrame(
    [
        {
            "method": "VADER (HW 3a baseline, same test split)",
            "trainable": "n/a (lexicon)",
            "epochs": 0,
            "accuracy": vader_acc,
            "f1": vader_f1,
        },
        {
            "method": "BERT frozen encoder + classifier head",
            "trainable": "classifier only",
            "epochs": 1,
            "accuracy": frozen_eval["eval_accuracy"],
            "f1": frozen_eval["eval_f1"],
        },
        {
            "method": "BERT full fine-tune (class lab 6a defaults)",
            "trainable": "all layers",
            "epochs": 1,
            "accuracy": lab_eval["eval_accuracy"],
            "f1": lab_eval["eval_f1"],
        },
    ]
)
compare["accuracy"] = compare["accuracy"].round(4)
compare["f1"] = compare["f1"].round(4)
display(compare)

best_acc = compare["accuracy"].max()
vader_row = compare.iloc[0]
ft_row = compare.iloc[2]
delta = ft_row["accuracy"] - vader_row["accuracy"]
print()
print(
    f"Full fine-tune accuracy {ft_row['accuracy']:.4f} vs VADER {vader_row['accuracy']:.4f} "
    f"({delta:+.4f} absolute, {100 * delta / vader_row['accuracy']:+.1f}% relative)."
)
if delta > 0:
    print("Yes — fine-tuning improved accuracy over the Homework 3a VADER baseline on the same test reviews.")
else:
    print("Fine-tuning did not beat VADER on this split; see the frozen-head row and Homework 6b for other Trainer settings.")


## Learnings (6a)

1. **Same task, different method:** Homework 3a used a lexicon (VADER). Class lab 6a fine-tunes a pretrained encoder (`bert-base-cased`) with a classification head using Hugging Face `Trainer`.
2. **Why fine-tuning helps:** BERT already encodes context, negation, and domain-ish phrasing. One epoch of labeled Amazon reviews adapts that representation to *this* product-review distribution, which VADER cannot do.
3. **Frozen vs full:** Training only the classifier is a useful ablation from the lab. Full fine-tuning usually wins because the encoder itself shifts toward the review domain.
4. **Fair comparison:** Accuracy is measured on a held-out test split, not on the training sentences.

Homework **6b** takes the same pipeline and sweeps `TrainingArguments` (learning rate, epochs, batch size, warmup, weight decay) to see if another setting improves **F1**.
